# RSICCformer: Colab eğitimi ve değerlendirmesi

Bu notebook kodu GitHub’dan indirir, LEVIR-CC verisini hazırlar ve eğitim, değerlendirme ve örnek çıkarım komutlarını çalıştırır. Kod ve veri `/content/RSICC` altında tutulur; checkpoint ve sonuçlar Google Drive’a kaydedilir.

Colab çalışma zamanı ayarlarından GPU seçin. Eğitim A100 üzerinde çalıştırılmıştır. Batch boyutu ve veri yükleme işçisi sayısı kullanılan donanıma göre ayarlanmalıdır.

Hücreleri sırayla çalıştırın. Repo adresi, branch veya commit, veri seti bağlantısı ve Drive yolları ilgili hücrelerde tanımlanır.

## 1. GPU ve Java kontrolü

METEOR metriği için Java gerekir.

In [ ]:
import torch

!nvidia-smi
!java -version

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    raise RuntimeError("Colab çalışma zamanı ayarlarından GPU seçin.")


## 2. Google Drive bağlantısı

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. GitHub’dan kodu indirme

`REPO_URL` ve `REPO_REF` değerlerini çalıştırılacak fork ve branch veya commit için ayarlayın. Kod `/content/RSICC` dizinine klonlanır. Bu hücreyi mevcut bir kod dizini üzerinde yeniden çalıştırmak yerine yeni bir çalışma zamanı başlatın.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/yakupgulcan/RSICC.git"
REPO_REF = "feature/modernize-runtime"  # Branch, tag veya commit hash
PROJECT_DIR = "/content/RSICC"

if os.path.exists(PROJECT_DIR):
    raise FileExistsError(f"{PROJECT_DIR} zaten mevcut. Yeni bir çalışma zamanı kullanın.")

subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=PROJECT_DIR, check=True)
os.chdir(PROJECT_DIR)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)


## 4. Bağımlılıklar ve ortam kaydı

Kurulumdan sonra ortam bilgilerini ve Git commitini kaydedin. Colab sürümleri oturumlar arasında değişebilir. Kurulum çalışma zamanını yeniden başlatmayı gerektirirse yeniden başlatın ve ortam kaydı hücresini tekrar çalıştırın.

In [ ]:
%pip install -r requirements.txt --quiet
%pip install gdown --quiet


In [ ]:
import json
import platform
import subprocess
import torch
import torchvision

environment = {
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "torchvision": torchvision.__version__,
    "pytorch_cuda_build": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
}
print(json.dumps(environment, indent=2))
ENVIRONMENT_PATH = "/content/drive/MyDrive/RSICC_Outputs/environment.json"
os.makedirs(os.path.dirname(ENVIRONMENT_PATH), exist_ok=True)
with open(ENVIRONMENT_PATH, "w") as f:
    json.dump(environment, f, indent=2)
print(f"Ortam kaydı: {ENVIRONMENT_PATH}")


## 5. LEVIR-CC veri setini indirme

`DATASET_DRIVE_URL` veri seti ZIP dosyasını göstermelidir. Açılan dizinde `LevirCCcaptions.json` ve `images/` bulunmalıdır. Caption dosyası mevcutsa indirme atlanır.

In [ ]:
DATASET_DRIVE_URL = "https://drive.google.com/file/d/1YgWES-0OOL-3KK4yIV3-y8MN-_c1ELEE/view?usp=drive_link"
DATASET_DIR = "/content/RSICC/Levir_CC_dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

# Caption dosyası mevcutsa indirmeyi atla.
if not os.path.exists(os.path.join(DATASET_DIR, "LevirCCcaptions.json")):
    print("Veri seti indiriliyor...")
    !gdown "{DATASET_DRIVE_URL}" -O /content/dataset_download.zip
    print("Zipten çıkarılıyor...")
    !unzip -q -o /content/dataset_download.zip -d "{DATASET_DIR}"
    !rm -f /content/dataset_download.zip
else:
    print("Levir_CC_dataset zaten mevcut, indirme adımı atlandı.")

!ls -lh "{DATASET_DIR}"

## 6. Veriyi ön işleme

`DRIVE_DATA_BACKUP` mevcutsa işlenmiş veri buradan açılır. Yoksa `create_input_files.py` HDF5 ve JSON dosyalarını üretir ve `data/` dizini Drive’a yedeklenir. Yedek, aynı veri seti ve ön işleme ayarlarıyla üretilmiş olmalıdır.

In [ ]:
import os

DATA_DIR = "/content/RSICC/data"
os.makedirs(DATA_DIR, exist_ok=True)

# İşlenmiş veri yedeği:
DRIVE_DATA_BACKUP = "/content/drive/MyDrive/RSICC/data_preprocessed.zip"

train_hdf5 = os.path.join(DATA_DIR, "TRAIN_IMAGES_LEVIR_CC_5_cap_per_img_5_min_word_freq.hdf5")

if not os.path.exists(train_hdf5):
    if os.path.exists(DRIVE_DATA_BACKUP):
        print("İşlenmiş veri yedeği açılıyor...")
        !unzip -q -o "{DRIVE_DATA_BACKUP}" -d /content/RSICC/
    else:
        print("data/ klasörü oluşturuluyor ve create_input_files.py çalıştırılıyor...")
        !python create_input_files.py
        
        # İşlenmiş veriyi sonraki oturumlar için yedekle.
        print("İşlenmiş veri Drive’a yedekleniyor...")
        !zip -q -r /content/data_preprocessed.zip ./data
        !cp /content/data_preprocessed.zip "{DRIVE_DATA_BACKUP}"
        !rm -f /content/data_preprocessed.zip
        print("Veri yedeği kaydedildi.")
else:
    print("data/ klasöründeki HDF5 dosyaları zaten hazır.")

!ls -lh ./data/

## 7. Model eğitimi

Örnek komut 20 epoch, batch boyutu 32 ve 4 veri yükleme işçisi kullanır. Bellek yetersizse batch boyutunu azaltın. Checkpointler `DRIVE_CHECKPOINT_DIR` dizinine yazılır.

In [ ]:
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/RSICC_Outputs/models_checkpoint/"
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

# Eğitim parametreleri:
!python train.py \
  --data_folder ./data/ \
  --data_name LEVIR_CC_5_cap_per_img_5_min_word_freq \
  --savepath "{DRIVE_CHECKPOINT_DIR}" \
  --epochs 20 \
  --batch_size 32 \
  --workers 4 \
  --encoder_lr 1e-4 \
  --decoder_lr 1e-4

## 8. Test setinde değerlendirme

`eval.py`, checkpoint dizinindeki tüm dosyaları sırayla yükler. Tek bir modeli değerlendirmek için yalnızca o checkpointi içeren bir dizin kullanın ve `EVAL_CHECKPOINT_DIR` değerini ayarlayın. Caption, referans ve skor JSON dosyaları Drive’a kopyalanır. Aynı çıktı adları her checkpoint için yeniden kullanılır.

In [ ]:
EVAL_CHECKPOINT_DIR = DRIVE_CHECKPOINT_DIR

!python eval.py \
  --data_folder ./data/ \
  --data_name LEVIR_CC_5_cap_per_img_5_min_word_freq \
  --path "{EVAL_CHECKPOINT_DIR}" \
  --Split TEST \
  --beam_size 1

# Değerlendirme sonuçlarını Drive’a kopyala.
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/RSICC_Outputs/eval_results/"
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
import os
if os.path.exists("./eval_results_fortest"):
    !cp -r ./eval_results_fortest/* "{DRIVE_RESULTS_DIR}"
    print("Değerlendirme sonuçları Drive’a kopyalandı.")
else:
    print("eval_results_fortest klasörü bulunamadı.")


## 9. Örnek görüntü çifti için caption üretme

`caption.py` de belirtilen dizindeki tüm checkpointleri işler ve aynı çıktı dosyasını yeniden kullanır.

In [ ]:
!python caption.py \
  --img_A ./Example/A/train_000016.png \
  --img_B ./Example/B/train_000016.png \
  --path "{EVAL_CHECKPOINT_DIR}" \
  --data_folder ./data/ \
  --data_name LEVIR_CC_5_cap_per_img_5_min_word_freq

!cat ./eval_results/*.json